### Install dependencies if running remotely with tools like Google Colab


In [ ]:
! pip install os
! pip install json
! pip install re
! pip install python-docx

### Imports

In [33]:
import os
import json
import re
import subprocess
from docx import Document

In [34]:
def extraer_pactos(doc_path):
    """
    Extrae los pactos de un documento .docx o .doc y los almacena con title y bodytext en JSON.
    
    Parámetros:
    - doc_path (str): Ruta al archivo .docx o .doc
    
    Retorna:
    - list[dict]: Lista de diccionarios con title y bodytext de cada pacto.
    """

    doc = Document(doc_path)
    pactos = []
    pacto_actual = None
    body_text = ""

    # Expresión regular para detectar títulos de pactos (masculinos, femeninos, con punto opcional)
    regex_pacto = re.compile(r"^(PRIMERO|PRIMERA|SEGUNDO|SEGUNDA|TERCERO|TERCERA|CUARTO|CUARTA|"
                             r"QUINTO|QUINTA|SEXTO|SEXTA|SÉPTIMO|SÉPTIMA|OCTAVO|OCTAVA|NOVENO|NOVENA|"
                             r"DÉCIMO|DÉCIMA|UNDÉCIMO|UNDÉCIMA|DUODÉCIMO|DUODÉCIMA|DECIMOTERCERO|DECIMOTERCERA|"
                             r"DECIMOCUARTO|DECIMOCUARTA|DECIMOQUINTO|DECIMOQUINTA|DECIMOSEXTO|DECIMOSEXTA|"
                             r"DECIMOSÉPTIMO|DECIMOSÉPTIMA|DECIMOCTAVO|DECIMOCTAVA|DECIMONOVENO|DECIMONOVENA|"
                             r"VIGÉSIMO|VIGÉSIMA|VIGESIMOPRIMERO|VIGESIMOPRIMERA|VIGESIMOSEGUNDO|VIGESIMOSEGUNDA|"
                             r"VIGESIMOTERCERO|VIGESIMOTERCERA|VIGESIMOCUARTO|VIGESIMOCUARTA|VIGESIMOQUINTO|VIGESIMOQUINTA|"
                             r"VIGESIMOSEXTO|VIGESIMOSEXTA|VIGESIMOSÉPTIMO|VIGESIMOSÉPTIMA|VIGESIMOOCTAVO|VIGESIMOOCTAVA|"
                             r"VIGESIMONOVENO|VIGESIMONOVENA|TRIGÉSIMO|TRIGÉSIMA)\.?", re.IGNORECASE)

    for paragraph in doc.paragraphs:
        texto = paragraph.text.strip()

        if not texto:
            continue  # Ignorar líneas vacías

        # Si el párrafo es un título de pacto, guardar el anterior y empezar uno nuevo
        if regex_pacto.match(texto):
            if pacto_actual:
                pactos.append({"title": pacto_actual, "bodytext": body_text.strip()})
            pacto_actual = texto.rstrip(".")  # Guardar título sin punto final
            body_text = ""  # Reiniciar el contenido
        elif pacto_actual:
            body_text += " " + texto

    # Guardar el último pacto si existe
    if pacto_actual:
        pactos.append({"title": pacto_actual, "bodytext": body_text.strip()})

    return pactos

In [35]:
def procesar_documentos(directorio):
    """
    Procesa todos los documentos .docx en un directorio y extrae los pactos en formato JSON.

    Parámetros:
    - directorio (str): Ruta de la carpeta con los documentos .docx
    """
    resultados = {}

    for archivo in os.listdir(directorio):
        if archivo.endswith(".docx"):
            ruta_completa = os.path.join(directorio, archivo)
            pactos = extraer_pactos(ruta_completa)
            resultados[archivo] = pactos

    # Guardar resultados en un archivo JSON
    with open("contratos.json", "w", encoding="utf-8") as json_file:
        json.dump(resultados, json_file, indent=4, ensure_ascii=False)

    print("✅ Los resultados están en 'contratos_pactos.json'.")



In [36]:
# Ruta donde están los archivos .docx
directorio_docx = "../dataset/contractExamples_org"

# Ejecutar la extracción
procesar_documentos(directorio_docx)

✅ Los resultados están en 'contratos_pactos.json'.
